# 加权百分位数的计算

> 对应代码：`MathStatsCode/code_in_notes/Appendix.B/weight_percentile.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：编写一个 r 类 Stata 程序 `weight_pctl`，根据给定的权重变量计算变量的加权百分位数（分位数），并用 CHFS 家庭收入数据演示其用法与结果。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Appendix.B`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Appendix.B")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**加权百分位数**

普通百分位数把每个观测看得同样重要；当数据带有**权重**（抽样权重或频数权重，如 CHFS 家庭数据中的家庭成员人数 `a1111`，将其作为频数权重即等价于按人口加权）时，每个观测代表的人群规模不同，应当使用**加权百分位数**：

- 记观测为 $x_i$、权重为 $w_i$，加权经验分布函数为 $F_w(x)=\sum_{x_i\le x}w_i\big/\sum_i w_i$；
- 第 $p$ 百分位数就是使 $F_w(x)\ge p/100$ 成立的**最小** $x$；
- 当所有权重相等时，它退化为通常的样本百分位数。

Stata 的 `summarize, detail` 配合 `fweight` 可以给出加权分位数，但若想把分位数结果保存下来、或按任意权重（非整数频数）计算，写一个程序更灵活。本程序 `weight_pctl` 的要点：

- `program weight_pctl, sortpreserve rclass`：定义 **r 类**程序（结果存放在 `r()` 中）；`sortpreserve` 让程序结束后自动恢复调用前的排序；
- `syntax varlist(max=1), weight(varname) p(numlist integer >0 <100 sort)`：只接受一个目标变量；`weight()` 指定权重变量；`p()` 接受 1–99 的整数列表并自动排序；
- `tempvar`/`tempname`/`local`：临时变量、临时名与局部宏，程序结束自动清理；
- 先按目标变量排序，再计算**标准化权重**及其**累积和**，得到加权经验分布；
- 对每个百分位 $p$，用 `while` 循环找到第一个累积权重不小于 $p/100$ 的观测，把该观测的变量值通过 `return scalar` 存为 `r(p25)` 等结果；
- `continue, break` 在找到结果后立即跳出 `while` 循环。

In [ ]:
%%stata
cap program drop weight_pctl
program weight_pctl, sortpreserve rclass
  version 15
  syntax varlist(max=1), weight(varname) p(numlist integer >0 <100 sort)
  tempvar cum_weight stand_weight sum_weight
  egen `sum_weight'=total(`weight') if `varlist'~=.
  gen `stand_weight'=`weight'/`sum_weight' if `varlist'~=.
  sort `varlist'
  gen `cum_weight'=`stand_weight' if _n==1
  replace `cum_weight'=`cum_weight'[_n-1]+`stand_weight'[_n] if _n>1
  tempname N i
  local `N'=_N
  local `i'=0
  foreach n of numlist `p'{
    while `++`i'' <= ``N''{
      if `cum_weight'[``i'']>=(`n'/100){
        return scalar p`n'=`varlist'[``i'']
        continue, break
      }
    }
  }
end

**载入数据并调用程序**

使用 CHFS 2017 家庭数据 `chfs2017_hh.dta`：`total_income` 为家庭总收入，`a1111` 为家庭成员人数，这里把它作为**频数权重**（相当于把每个家庭按其成员数重复），从而得到按人口加权的收入分布。先用 `drop if a1111>=.` 删除成员人数缺失的观测，避免标准化权重之和出错；然后调用 `weight_pctl total_income, weight(a1111) p(25 33 50 67 75)` 计算第 25、33、50、67、75 百分位数。

程序把结果依次存放在 `r(p25)`、`r(p33)`、`r(p50)`、`r(p67)`、`r(p75)` 中；`di r(p25) _skip r(p33) _skip ...` 用 `_skip` 在同一行间隔输出，便于逐项核对。

In [ ]:
%%stata
use "../datasets/chfs2017_hh.dta", clear
drop if a1111>=.
weight_pctl total_income, weight(a1111) p(25 33 50 67 75)
di r(p25) _skip r(p33) _skip r(p50) _skip r(p67) _skip r(p75)

**与 `summarize, detail` 的加权结果对照**

`su total_income [fweight=a1111], de` 把 `a1111` 当作频数权重，Stata 会按加权样本给出均值、标准差和百分位数。可以看到，`su` 报告的 25%、50%、75% 分位数与程序返回的 `r(p25)`、`r(p50)`、`r(p75)` 完全一致，说明加权经验分布的计算是正确的（33%、67% 是程序中额外计算的分位数）。

In [ ]:
%%stata
su total_income [fweight=a1111], de